# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ksusmitha879-cyber/FlyrankStarter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**My answer: Logistic Regression → Decision Tree → Random Forest, in that order, on an
observed yes/no label.**

Week 4's rule baseline was graded against `is_declining_label = (trend_direction == "down")`,
which is an **observed** outcome (period-over-period impressions), not a proxy I invented. Per
the `training-honest-models` method table, "yes/no with an observed label" starts with
Logistic Regression, then Random Forest — so that is the ladder I climb, adding a Decision
Tree in between because it is the one model I can print and read end-to-end, which the skill
explicitly values ("a depth-2 decision tree you can print and read teaches more than an opaque
model 2 points stronger"). I add complexity (tree → forest) only because the comparison table
in Section 3 can tell me whether it earns its keep — if the extra complexity doesn't beat the
simpler model on the same split and metric, I say so instead of picking the fanciest one by
default.

I did **not** use K-Means/clustering here: my lane's decision ("which page should a reviewer
open first") is already a ranking/scoring question with an available observed label to check
scores against, so a supervised classifier evaluated by precision@K is a more direct fit than
an unsupervised grouping I would have to re-interpret as a proxy for the same thing.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd
from pathlib import Path
import os

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Download the data file if it doesn't exist in /content/ (Colab)
if not os.path.exists("/content/content_refresh_anonymized.csv"):
    if not os.path.exists("../../data/raw/content_refresh_anonymized.csv"):
        os.system(
            "wget -q https://raw.githubusercontent.com/ksusmitha879-cyber/FlyrankStarter/main/"
            "data/raw/content_refresh_anonymized.csv -P /content/"
        )

# Path resolution: works in Colab (after downloading to /content) and in a local clone
candidates = [
    Path("/content/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("data/raw/content_refresh_anonymized.csv"),
]
DATA_PATH = next((p for p in candidates if p.exists()), candidates[-1])
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} rows x {df.shape[1]} cols from {DATA_PATH}")

# Eligible pool -- SAME definition as w04_baseline_score.ipynb, so the pool being scored
# is identical to the Week-4 baseline's pool (avg_position==0 means "no data", not rank zero).
valid = df[df["avg_position"] > 0].copy()
visible = valid[valid["impressions_90d"] >= 100].copy()
print(f"Eligible pool (avg_position>0 & impressions_90d>=100): {len(visible):,} of {len(df):,} rows")

# Target -- observed, derived from trend_direction (never used as a feature -- see the
# leakage assertion in Section 3). This is the SAME label Week 4's baseline was graded against.
visible["is_declining_label"] = (visible["trend_direction"] == "down").astype(int)

Loaded 30,000 rows x 44 cols from /content/content_refresh_anonymized.csv
Eligible pool (avg_position>0 & impressions_90d>=100): 22,006 of 30,000 rows


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**My answer: grouped by `client_id` (GroupShuffleSplit, 20% of clients held out) as the
primary split — with a naive stratified-random split computed alongside it as an honesty
check, per the `hunting-leakage-and-validating` skill's own advice to "report the random-split
number next to the honest-split number... the GAP between them is itself a finding."**

There is no timestamp per row in this starter CSV (it's a single trailing-90-day snapshot), so
a time-based split isn't available here — that is a real limitation of the starter dataset. But
`client_id` clearly repeats (30 clients, hundreds of pages each), and this internship's own
data dictionary flags `client_id` as the column to use for grouped train/test splits.
A random row split would let the model learn "pages that look like client X's pages tend to be
declining/not" and get credit for memorizing a client's overall style rather than a pattern that
generalizes to a page from a client it has never seen — which is the realistic deployment
question ("would this work on a new client's content?").

I compute both splits below and carry both into Section 3's comparison table.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit, train_test_split

target = visible["is_declining_label"]

def grouped_split(frame, y, group_col="client_id", test_size=0.2, seed=RANDOM_STATE):
    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    train_idx, test_idx = next(splitter.split(frame, y, groups=frame[group_col]))
    return train_idx, test_idx

def stratified_split(frame, y, test_size=0.2, seed=RANDOM_STATE):
    idx = np.arange(len(frame))
    train_idx, test_idx = train_test_split(idx, test_size=test_size, random_state=seed, stratify=y)
    return train_idx, test_idx

grp_train, grp_test = grouped_split(visible, target)
row_train, row_test = stratified_split(visible, target)

print("Split design check (both splits, same eligible pool, same seed=42):\n")
for name, (tr, te) in {
    "grouped_by_client (primary)": (grp_train, grp_test),
    "stratified_random (honesty check)": (row_train, row_test),
}.items():
    tr_clients = set(visible.iloc[tr]["client_id"])
    te_clients = set(visible.iloc[te]["client_id"])
    overlap = tr_clients & te_clients
    print(
        f"{name}: train={len(tr):,}  test={len(te):,}  "
        f"train_base_rate={target.iloc[tr].mean():.3f}  test_base_rate={target.iloc[te].mean():.3f}  "
        f"client_overlap_between_train_and_test={len(overlap)}"
    )
print("\nThe grouped split has ZERO client overlap by construction -- a page in test comes")
print("from a client the model never trained on. The stratified split shares clients across")
print("train/test on purpose, as the naive comparison point.")

Split design check (both splits, same eligible pool, same seed=42):

grouped_by_client (primary): train=18,392  test=3,614  train_base_rate=0.607  test_base_rate=0.553  client_overlap_between_train_and_test=0
stratified_random (honesty check): train=17,604  test=4,402  train_base_rate=0.598  test_base_rate=0.598  client_overlap_between_train_and_test=28

The grouped split has ZERO client overlap by construction -- a page in test comes
from a client the model never trained on. The stratified split shares clients across
train/test on purpose, as the naive comparison point.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**Method:** Logistic Regression, Decision Tree, Random Forest, all with `class_weight="balanced"`
and `random_state=42`, trained on the same eligible pool as Week 4. Features come straight from
`scripts/ml_utils.py`'s `MODEL_NUMERIC_FEATURES` / `MODEL_CATEGORICAL_FEATURES` (the repo's own
definition of what's safe to model with), plus two `has_*` missingness flags per the data
dictionary's warning that missingness follows `content_type` — a blind `fillna(0)` would
silently encode content type into the numbers, so I flag it instead. `trend_direction`,
`trend_pct`, and `is_declining_label` never enter the feature set (asserted in code below);
neither do the pseudonymous IDs or `provider_used`/`model_used`, which the dictionary marks
"Not a model feature."

**Baseline, recomputed fairly:** Week 4's rule (`ctr_gap × log1p(impressions) × (1 + stale_flag)`)
used `position_tier` median CTR computed over the *whole* eligible pool. To compare baseline vs
model on the exact same footing, I recompute that rule's tier medians from the **train** rows
only and apply them to the test rows — so neither the baseline nor the models see any test-set
statistic before they're scored. Same test rows, same metrics (precision@10/50/100, ROC AUC,
average precision, base rate) go into one table for both.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
visible["log_impressions_90d"] = np.log1p(visible["impressions_90d"])
visible["log_clicks_90d"] = np.log1p(visible["clicks_90d"])
visible["log_sessions_90d"] = np.log1p(visible["sessions_90d"])
visible["log_ai_sessions_90d"] = np.log1p(visible["ai_sessions_90d"])
visible["has_word_count"] = visible["word_count"].notna().astype(int)
visible["has_keyword_data"] = visible["search_volume"].notna().astype(int)

NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct", "has_word_count", "has_keyword_data",
]
CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]

# Leakage assertion -- mirrors the same check w04 ran on the rule score.
LABEL_AND_ID_COLS = {"trend_direction", "trend_pct", "is_declining_label", "content_id", "client_id",
                      "provider_used", "model_used"}
assert set(NUMERIC_FEATURES + CATEGORICAL_FEATURES).isdisjoint(LABEL_AND_ID_COLS), \
    "feature set must not touch label-derived columns or IDs"
print(f"Leakage check passed. Feature count before encoding: {len(NUMERIC_FEATURES) + len(CATEGORICAL_FEATURES)}")

for c in NUMERIC_FEATURES:
    visible[c] = pd.to_numeric(visible[c], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
for c in CATEGORICAL_FEATURES:
    visible[c] = visible[c].fillna("unknown").astype(str)

Leakage check passed. Feature count before encoding: 28


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean())

def baseline_score_fit_apply(train_frame, apply_frame):
    """Week 4's rule score, refit honestly: tier medians and the CTR-gap rule are learned
    from TRAIN only, then applied to whichever frame we are scoring."""
    tier_median_ctr = train_frame.groupby("position_tier")["ctr"].median()
    global_median_ctr = train_frame["ctr"].median()
    tier_lookup = apply_frame["position_tier"].map(tier_median_ctr).fillna(global_median_ctr)
    ctr_gap = (tier_lookup - apply_frame["ctr"]).clip(lower=0)
    stale_flag = (apply_frame["days_since_last_update"] >= 90).astype(int)
    return ctr_gap * np.log1p(apply_frame["impressions_90d"]) * (1 + stale_flag)

def build_preprocessor():
    return ColumnTransformer([
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ])

def build_models():
    return {
        "logistic_regression": Pipeline([
            ("prep", build_preprocessor()),
            ("model", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE)),
        ]),
        "decision_tree": Pipeline([
            ("prep", build_preprocessor()),
            ("model", DecisionTreeClassifier(class_weight="balanced", max_depth=5, min_samples_leaf=50, random_state=RANDOM_STATE)),
        ]),
        "random_forest": Pipeline([
            ("prep", build_preprocessor()),
            ("model", RandomForestClassifier(class_weight="balanced_subsample", n_estimators=300,
                                              max_depth=10, min_samples_leaf=25, n_jobs=-1, random_state=RANDOM_STATE)),
        ]),
    }

def evaluate_split(frame, y, train_idx, test_idx, split_name):
    train_frame, test_frame = frame.iloc[train_idx], frame.iloc[test_idx]
    train_y, test_y = y.iloc[train_idx], y.iloc[test_idx]

    baseline_scores = baseline_score_fit_apply(train_frame, test_frame)
    rows = [{
        "split": split_name, "method": "baseline_rule (Week 4)",
        "roc_auc": roc_auc_score(test_y, baseline_scores),
        "avg_precision": average_precision_score(test_y, baseline_scores),
        "precision_at_10": precision_at_k(test_y, baseline_scores, 10),
        "precision_at_50": precision_at_k(test_y, baseline_scores, 50),
        "precision_at_100": precision_at_k(test_y, baseline_scores, 100),
        "base_rate": test_y.mean(),
    }]
    for name, model in build_models().items():
        model.fit(train_frame[NUMERIC_FEATURES + CATEGORICAL_FEATURES], train_y)
        proba = model.predict_proba(test_frame[NUMERIC_FEATURES + CATEGORICAL_FEATURES])[:, 1]
        rows.append({
            "split": split_name, "method": name,
            "roc_auc": roc_auc_score(test_y, proba),
            "avg_precision": average_precision_score(test_y, proba),
            "precision_at_10": precision_at_k(test_y, proba, 10),
            "precision_at_50": precision_at_k(test_y, proba, 50),
            "precision_at_100": precision_at_k(test_y, proba, 100),
            "base_rate": test_y.mean(),
        })
    return pd.DataFrame(rows)

results_grouped = evaluate_split(visible, target, grp_train, grp_test, "grouped_by_client")
results_random = evaluate_split(visible, target, row_train, row_test, "stratified_random")
comparison = pd.concat([results_grouped, results_random], ignore_index=True)

pd.set_option("display.width", 140)
print("MODEL vs BASELINE -- same eligible pool, same metrics, two split designs:\n")
print(comparison.round(3).to_string(index=False))

MODEL vs BASELINE -- same eligible pool, same metrics, two split designs:

            split                 method  roc_auc  avg_precision  precision_at_10  precision_at_50  precision_at_100  base_rate
grouped_by_client baseline_rule (Week 4)    0.598          0.630              0.6             0.78              0.75      0.553
grouped_by_client    logistic_regression    0.604          0.641              0.9             0.76              0.73      0.553
grouped_by_client          decision_tree    0.561          0.611              0.7             0.58              0.63      0.553
grouped_by_client          random_forest    0.594          0.633              0.7             0.74              0.69      0.553
stratified_random baseline_rule (Week 4)    0.572          0.659              0.6             0.66              0.72      0.598
stratified_random    logistic_regression    0.715          0.784              0.9             0.90              0.93      0.598
stratified_random          de

**Reading the table:** on the honest **grouped-by-client** split, no model clearly beats the
Week-4 rule baseline. Logistic Regression edges the baseline on average precision (0.641 vs
0.630) and precision@10 (0.90 vs 0.60), but the baseline still wins at precision@50 (0.78 vs
0.76) and precision@100 (0.75 vs 0.73) — a mixed result, not a clean win, and exactly the kind
of "report both, that IS the finding" case the `training-honest-models` skill calls out. The
Decision Tree and Random Forest do not beat the baseline on this split either; added complexity
does not pay for itself here.

The **stratified-random** split tells a very different, much rosier story (Random Forest ROC
AUC 0.756 vs baseline 0.572). That gap between the two splits is itself the most important
finding in this notebook: a random row split lets every model partly memorize "what a
client_X page tends to look like," because most clients appear on both sides of that split.
The grouped split removes that shortcut, and the apparent lift almost disappears — a sign that
most of the random-split "skill" was memorization, not a generalizable pattern. I'm treating the
grouped-split numbers as the honest answer to "does this beat the baseline," and the random
numbers as a documented cautionary comparison, not a result to act on.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.inspection import permutation_importance

train_frame, test_frame = visible.iloc[grp_train], visible.iloc[grp_test]
train_y, test_y = target.iloc[grp_train], target.iloc[grp_test]

lr_pipeline = build_models()["logistic_regression"]
lr_pipeline.fit(train_frame[NUMERIC_FEATURES + CATEGORICAL_FEATURES], train_y)
lr_proba = lr_pipeline.predict_proba(test_frame[NUMERIC_FEATURES + CATEGORICAL_FEATURES])[:, 1]

rf_pipeline = build_models()["random_forest"]
rf_pipeline.fit(train_frame[NUMERIC_FEATURES + CATEGORICAL_FEATURES], train_y)

# Permutation importance (menu item) on the RANDOM FOREST, scored on average precision,
# computed on the held-out grouped test set -- shuffles each column and reads the score drop,
# so it is a fit-then-checked importance, not just "what the tree happened to split on."
perm = permutation_importance(
    rf_pipeline, test_frame[NUMERIC_FEATURES + CATEGORICAL_FEATURES], test_y,
    n_repeats=10, random_state=RANDOM_STATE, scoring="average_precision", n_jobs=-1,
)
perm_table = pd.DataFrame({
    "feature": NUMERIC_FEATURES + CATEGORICAL_FEATURES,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)
print("Top 10 permutation importances (random_forest, avg_precision, grouped test set):\n")
print(perm_table.head(10).round(4).to_string(index=False))

Top 10 permutation importances (random_forest, avg_precision, grouped test set):

              feature  importance_mean  importance_std
                  ctr           0.0114          0.0032
       log_clicks_90d           0.0092          0.0021
          scroll_rate           0.0080          0.0017
         avg_position           0.0055          0.0014
days_with_impressions           0.0041          0.0012
        position_tier           0.0039          0.0015
     content_age_days           0.0033          0.0048
      engagement_rate           0.0030          0.0013
   days_with_sessions           0.0029          0.0013
             age_tier           0.0019          0.0039


**Sanity check on the top features:** `ctr`, `log_clicks_90d`, `scroll_rate`, and
`avg_position` lead — all plausible drivers of "is this page's traffic currently trending
down," and none of them is `trend_direction`/`trend_pct` or a sibling of the label (the
leakage assertion in Section 3 already guarantees that). No single feature towers over the
rest the way the leakage skill warns about ("one feature towers over all others, score is
near-perfect") — the top importance is around 0.01, not 0.5+ — so this doesn't look like a
disguised leak, just a genuinely hard, noisy problem.

In [ ]:
errs = test_frame.copy()
errs["y_true"] = test_y.values
errs["proba"] = lr_proba
errs["pred"] = (errs["proba"] >= 0.5).astype(int)
errs["error_type"] = np.select(
    [(errs.pred == 1) & (errs.y_true == 0), (errs.pred == 0) & (errs.y_true == 1)],
    ["false_positive", "false_negative"], default="correct",
)

print("Error rate by position_tier (logistic_regression, grouped test set):")
print(errs.groupby("position_tier")["error_type"].apply(lambda s: (s != "correct").mean()).round(3))
print("\nError rate by content_type:")
print(errs.groupby("content_type")["error_type"].apply(lambda s: (s != "correct").mean()).round(3))

cols = ["content_id", "proba", "position_tier", "content_type", "ctr", "avg_position", "days_since_last_update"]
print("\n3 most confident WRONG false positives (model said declining, it was not):")
print(errs[errs.error_type == "false_positive"].sort_values("proba", ascending=False).head(3)[cols].to_string(index=False))

print("\n3 most confident WRONG false negatives (model said fine, it was declining):")
print(errs[errs.error_type == "false_negative"].sort_values("proba", ascending=True).head(3)[cols].to_string(index=False))

Error rate by position_tier (logistic_regression, grouped test set):
position_tier
deep        0.413
page_1      0.422
page_3_5    0.474
striking    0.473
top_3       0.267
Name: error_type, dtype: float64

Error rate by content_type:
content_type
comparison article    0.361
feedly article        0.516
keyword article       0.453
Name: error_type, dtype: float64

3 most confident WRONG false positives (model said declining, it was not):
          content_id    proba position_tier       content_type  ctr  avg_position  days_since_last_update
content_1158aa50a684 0.961537        page_1 comparison article  0.0           7.2                      20
content_ab27c30d81f4 0.952172        page_1    keyword article  0.0           8.9                     304
content_619acf4bbcc3 0.951530        page_1 comparison article  0.0           7.2                       8

3 most confident WRONG false negatives (model said fine, it was declining):
          content_id    proba position_tier    content_typ

**What the errors look like:** the model is worst in the messy middle tiers (`page_3_5`,
`striking`) and best at the extremes (`top_3`) — the middle of the position ranking is exactly
where "is this page declining" is hardest to call from snapshot metrics alone, which matches
the mixed precision@K story above. By content type, `feedly article` has the highest error
rate; these are syndicated/aggregated pieces that the data dictionary already flags as missing
most of their keyword-context columns, so the model has the least to work with there.

The most confident false positives are `page_1` pages with `ctr = 0.0` and low
`days_since_last_update` — the model reads "zero clicks despite decent position" as a strong
decline signal, but these pages simply hadn't accumulated any clicks yet in the window and
were *not* independently flagged as declining. The most confident false negatives are
`deep`-tier pages (position 60–80) that the model was very sure were *not* declining, but
were — likely because at that far a position, `ctr`/`avg_position` carry almost no
discriminating signal (almost nothing gets clicked from position 70 either way), so the model
has little to distinguish a "declining deep page" from a "flat deep page." In both failure
patterns the model leans hardest on `ctr`/`avg_position`, and those two signals genuinely run
out of information at the tails of the ranking — a limitation of the feature set, not a bug in
the model.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.